# ⛪ IBPM CR - FASE 1: Varredura Completa & Plano Mestre de Mídia

> **REGRA DA FASE 1:** NENHUM VÍDEO É RENDERIZADO OU CORTADO NESTA ETAPA.
> **Objetivo:** Ingestão de dados de 100% do acervo (do 1º ao mais recente vídeo do canal `@ibpmcr7976`), transcrição leve em lote via `youtube-transcript-api` + `Faster-Whisper` na GPU T4, mineração de PNL (`spaCy`) e geração do **Plano Mestre de Mídia** (`plano_mestre_ibpmcr.json` e `SQLite`).

In [ ]:
# ============================================================
# 1. MONTAGEM DO DRIVE, INSTALAÇÃO DE DEPENDÊNCIAS E NODEJS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/matheusbarbosatech/ibpmcr-automation-system.git
%cd ibpmcr-automation-system

# Instala Node.js para suporte a JavaScript Runtime no yt-dlp
!apt-get update -qq && apt-get install -y -qq nodejs

# Instala pacotes de transcrição ultra-rápida
!pip install -q -U yt-dlp youtube-transcript-api
!pip install -q -r requirements.txt
!python -m spacy download pt_core_news_sm
!python config/setup_drive.py

print('✅ Ambiente da Fase 1 pronto com transcrição ultra-rápida!')

In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO DE CHAVES DE API (.env)
# ============================================================
import os
from dotenv import load_dotenv
load_dotenv(override=True)

print('✅ Configuração de ambiente inicializada!')

In [ ]:
# ============================================================
# 3. EXECUÇÃO DO FLUXO DE MAPEAMENTO & PLANO MESTRE
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv
load_dotenv(override=True)

from src.discovery.channel_sweeper import ChannelSweeper
from src.discovery.transcriber_batch import BatchTranscriber
from src.discovery.content_analyzer import ContentAnalyzer
from src.core.state_manager import MasterPlanManager
from src.discovery.generate_report import Phase1ReportGenerator

print('🔍 STEP 1: Varredura do Acervo Histórico (do 1º vídeo ao mais recente)...')
sweeper = ChannelSweeper()
catalog = sweeper.sweep_channel_metadata(limit=500)
print(f'✅ Total de vídeos mapeados no acervo: {len(catalog)}')
if catalog:
    print(f'📅 1º Vídeo Histórico (Mais Antigo): {catalog[0]["titulo_original"]} ({catalog[0]["data_publicacao"][:10]})')
    print(f'📅 Vídeo Mais Recente: {catalog[-1]["titulo_original"]} ({catalog[-1]["data_publicacao"][:10]})')

print('🎙️ STEP 2: Ingestão de Áudio e Transcrição em Lote...')
transcriber = BatchTranscriber()
analyzer = ContentAnalyzer()
master_mgr = MasterPlanManager()

# Processa o acervo em ordem cronológica (do 1º vídeo de 2022 ao mais recente)
for i, vid in enumerate(catalog, 1):
    data_str = vid['data_publicacao'][:10] if vid.get('data_publicacao') else ''
    print(f'[{i}/{len(catalog)}] -> Mapeando vídeo: {vid["video_id"]} - {vid["titulo_original"]} ({data_str})')
    trans_res = transcriber.get_video_transcription(vid['video_id'], vid['url'])
    analysis = analyzer.analyze_transcript(trans_res)
    master_mgr.update_video_analysis(vid['video_id'], vid, analysis)

print('📊 STEP 3: Gerando Relatórios Diagnósticos (PDF & HTML)...')
rep_gen = Phase1ReportGenerator()
reports = rep_gen.generate_diagnostic_reports()
print('🎉 FASE 1 CONCLUÍDA COM SUCESSO!')
print('Relatórios salvos em:', reports)